# 골든배치 1차 후보 검증

현재 확인 가능한 총수확량 기준 1차 후보 15개(RC·OC·APC 각 5개)가 같은 전략의 나머지 정상 배치보다 공정 안정성도 좋은지 탐색한다. 최종 골든배치 검증이 아니며 CSV는 저장하지 않는다.

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display
from scipy import stats


def find_project_root(start=Path.cwd()):
    for root in (start, *start.parents):
        if (root / 'data/interim/merged_data_ko.csv').exists():
            return root
    raise FileNotFoundError('merged_data_ko.csv를 찾을 수 없습니다.')


def fdr_bh(p_values):
    p_values = np.asarray(p_values, dtype=float)
    order = np.argsort(p_values)
    ranked = p_values[order]
    adjusted = ranked * len(ranked) / np.arange(1, len(ranked) + 1)
    adjusted = np.minimum.accumulate(adjusted[::-1])[::-1]
    result = np.empty_like(adjusted)
    result[order] = np.clip(adjusted, 0, 1)
    return result


def hedges_g(sample_a, sample_b):
    sample_a, sample_b = np.asarray(sample_a), np.asarray(sample_b)
    n_a, n_b = len(sample_a), len(sample_b)
    pooled_var = ((n_a - 1) * sample_a.var(ddof=1) + (n_b - 1) * sample_b.var(ddof=1)) / (n_a + n_b - 2)
    correction = 1 - 3 / (4 * (n_a + n_b) - 9)
    return correction * (sample_a.mean() - sample_b.mean()) / np.sqrt(pooled_var)


candidate_ids = [8, 9, 12, 17, 26, 35, 52, 55, 57, 60, 63, 66, 79, 83, 85]
data = pd.read_csv(find_project_root() / 'data/interim/merged_data_ko.csv')
data = data.loc[data['배치번호'] <= 90].copy()
print('1차 후보:', candidate_ids)

1차 후보: [8, 9, 12, 17, 26, 35, 52, 55, 57, 60, 63, 66, 79, 83, 85]


### 확인

사용한 후보는 RC 8·9·12·17·26, OC 35·52·55·57·60, APC 63·66·79·83·85다. 이 목록은 총수확량으로 고른 1차 후보이므로 확정 골든배치로 해석하지 않는다.

In [2]:
def slope(x, y):
    return float(np.polyfit(x, y, 1)[0])


rows = []
for batch_number, batch in data.groupby('배치번호', sort=True):
    batch = batch.sort_values('발효시간(h)')
    time = batch['발효시간(h)'].to_numpy()
    late = time / time[-1] >= 0.8
    our = batch['산소소모율(g/min)'].to_numpy()
    rows.append({
        '배치번호': batch_number,
        '전략': 'RC' if batch_number <= 30 else ('OC' if batch_number <= 60 else 'APC'),
        '후보': batch_number in candidate_ids,
        'pH표준편차': batch['pH'].std(ddof=1),
        'DO최솟값': batch['용존산소(mg/L)'].min(),
        'OUR변동계수': our.std(ddof=1) / abs(our.mean()),
        'CO2최댓값': batch['배가스이산화탄소(%)'].max(),
        '기질후기기울기': slope(time[late], batch.loc[late, '기질농도(g/L)']),
        '온도표준편차': batch['발효온도(K)'].std(ddof=1),
        '산총투입량': np.trapezoid(batch['산투입유량(L/h)'], time),
        '염기총투입량': np.trapezoid(batch['염기투입유량(L/h)'], time),
    })
batch_metrics = pd.DataFrame(rows)
display(batch_metrics.groupby(['전략', '후보']).size().unstack())

후보,False,True
전략,,
APC,25,5
OC,25,5
RC,25,5


### 확인

각 전략 안에서 후보 5개와 비후보 25개를 비교한다. 후보 선정에 사용된 총수확량과 그에 가까운 결과지표는 순환 검증을 피하기 위해 검정 대상에서 제외했다.

In [3]:
metrics = ['pH표준편차', 'DO최솟값', 'OUR변동계수', 'CO2최댓값', '기질후기기울기',
           '온도표준편차', '산총투입량', '염기총투입량']
standardized = batch_metrics.copy()
for metric in metrics:
    standardized[metric] = standardized.groupby('전략')[metric].transform(
        lambda values: (values - values.mean()) / values.std(ddof=1)
    )

rows = []
for metric in metrics:
    candidate = standardized.loc[standardized['후보'], metric].dropna().to_numpy()
    other = standardized.loc[~standardized['후보'], metric].dropna().to_numpy()
    welch = stats.ttest_ind(candidate, other, equal_var=False)
    mann = stats.mannwhitneyu(candidate, other, alternative='two-sided')
    rows.append({
        '지표': metric, '후보_z평균': candidate.mean(), '비후보_z평균': other.mean(),
        '평균차이': candidate.mean() - other.mean(),
        'Welch_p': welch.pvalue, 'Mann_p': mann.pvalue,
        'Hedges_g': hedges_g(candidate, other),
    })
candidate_results = pd.DataFrame(rows)
candidate_results['Welch_FDR'] = fdr_bh(candidate_results['Welch_p'])
candidate_results['Mann_FDR'] = fdr_bh(candidate_results['Mann_p'])
display(candidate_results.sort_values('Welch_FDR').round(6))

,지표,후보_z평균,비후보_z평균,평균차이,Welch_p,Mann_p,Hedges_g,Welch_FDR,Mann_FDR
1,DO최솟값,0.414595,-0.082919,0.497514,0.002348,0.042910,0.505152,0.018788,0.114428
0,pH표준편차,-0.287342,0.057468,-0.344810,0.008285,0.027963,-0.346797,0.033140,0.114428
4,기질후기기울기,-0.430147,0.086029,-0.516177,0.084789,0.028744,-0.524842,0.226104,0.114428
2,OUR변동계수,-0.299810,0.059962,-0.359772,0.264216,0.138008,-0.362124,0.376490,0.220813
3,CO2최댓값,0.216873,-0.043375,0.260247,0.282367,0.410606,0.260774,0.376490,0.547474
6,산총투입량,-0.310947,0.062189,-0.373137,0.196722,0.106708,-0.375846,0.376490,0.213415
7,염기총투입량,-0.045410,0.009082,-0.054492,0.831081,0.811737,-0.054347,0.949807,0.879524
5,온도표준편차,-0.006633,0.001327,-0.007959,0.974115,0.879524,-0.007936,0.974115,0.879524


### 최종 판단

- 전략 내 표준화 후 후보는 비후보보다 DO 최솟값이 높았다(z 평균차이 +0.498, Welch FDR=0.0188). 산소 부족을 덜 겪는 방향이다.
- 후보의 pH 표준편차도 더 낮았다(z 평균차이 -0.345, Welch FDR=0.0331). pH 운전이 더 안정적인 방향이다.
- 그러나 두 지표 모두 Mann–Whitney FDR은 0.05를 넘었다. 후보가 15개뿐이고 분포 차이에 민감하므로 강한 확증이 아니라 유망한 신호로 본다.
- OUR 변동, CO2 최대, 후기 기질 기울기, 온도 변동, 산·염기 총투입량에는 FDR 기준 차이 근거가 부족했다.
- 이 결과로 후보를 확정하면 안 된다. 최종 골든배치 ID가 정해지면 동일 검정을 다시 수행하고, 가능하면 후보 선정에 쓰지 않은 독립 배치 또는 교차검증으로 재현성을 확인해야 한다.